In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

xl = pd.ExcelFile('/Users/headofthetable/inventory-optimisation/data/raw/Apparel_Store_Inventory_JIT_Dataset.xlsx')
sales = pd.read_excel(xl, 'Daily_Sales_Transactions', header=3)
products = pd.read_excel(xl, 'Product_Master', header=3)

sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)

print("Loaded:", sales.shape)

Loaded: (45423, 11)


In [2]:
# Merge sales with product master to get sub-category
sales = sales.merge(products[['SKU', 'Sub-Category', 'Gender Category']], 
                    on='SKU', how='left')

# Add month and year columns
sales['Month'] = sales['Date'].dt.month
sales['Year'] = sales['Date'].dt.year

# Aggregate to monthly sub-category level
monthly = (sales.groupby(['Year', 'Month', 'Sub-Category'])['Units Sold']
           .sum()
           .reset_index())

monthly.columns = ['Year', 'Month', 'Sub_Category', 'Units_Sold']

print("Monthly aggregated shape:", monthly.shape)
print()
print("Sub-categories:", monthly['Sub_Category'].unique())
print()
print(monthly.head(10))

Monthly aggregated shape: (102, 4)

Sub-categories: <ArrowStringArray>
['Casual Shirts',       'Dresses',  'Ethnic Kurta',  'Ethnic Kurti',
   'Ethnic Wear', 'Formal Shirts',       'Jackets',         'Jeans',
      'Leggings',        'Shirts',        'Shorts',        'Skirts',
      'Sweaters',   'Sweatshirts',      'T-Shirts',          'Tops',
      'Trousers']
Length: 17, dtype: str

   Year  Month   Sub_Category  Units_Sold
0  2026      1  Casual Shirts         207
1  2026      1        Dresses        1702
2  2026      1   Ethnic Kurta         238
3  2026      1   Ethnic Kurti         423
4  2026      1    Ethnic Wear         822
5  2026      1  Formal Shirts        1423
6  2026      1        Jackets        1493
7  2026      1          Jeans         990
8  2026      1       Leggings         428
9  2026      1         Shirts         455


In [3]:
tshirts = monthly[monthly['Sub_Category'] == 'T-Shirts'].copy()
tshirts = tshirts.sort_values('Month').reset_index(drop=True)

print("T-Shirts monthly sales:")
print(tshirts)

T-Shirts monthly sales:
   Year  Month Sub_Category  Units_Sold
0  2026      1     T-Shirts        1964
1  2026      2     T-Shirts        1363
2  2026      3     T-Shirts        1571
3  2026      4     T-Shirts        1007
4  2026      5     T-Shirts        1087
5  2026      6     T-Shirts        1373


In [4]:
# Check what brand column is called in products
print(products.columns.tolist())

['SKU', 'Style Code', 'Product Name', 'Gender Category', 'Sub-Category', 'Brand Line', 'Color', 'Size', 'Fabric', 'Fit Type', 'Season Tag', 'MRP (INR)', 'Cost Price (INR)', 'Margin %', 'Supplier', 'Lead Time (Days)', 'MOQ (Units)', 'Launch Date', 'Velocity Tier']


In [5]:
print("Brand lines:", products['Brand Line'].unique())
print("Season tags:", products['Season Tag'].unique())
print("Velocity tiers:", products['Velocity Tier'].unique())
print()

# How many T-Shirt SKUs per brand
tshirt_skus = products[products['Sub-Category'] == 'T-Shirts']
print("T-Shirt SKUs per brand:")
print(tshirt_skus['Brand Line'].value_counts())

Brand lines: <ArrowStringArray>
[   'FlexDenim', 'Urban Basics',   'StreetEdge',    'EliteWear',
     'CasualCo',   'ComfortFit']
Length: 6, dtype: str
Season tags: <ArrowStringArray>
['Summer', 'Winter', 'All Season', 'Festive/Ethnic']
Length: 4, dtype: str
Velocity tiers: <ArrowStringArray>
['Medium Mover', 'Fast Mover', 'Slow Mover']
Length: 3, dtype: str

T-Shirt SKUs per brand:
Brand Line
FlexDenim    42
CasualCo     24
Name: count, dtype: int64


In [6]:
# Create sales_full first
sales_full = sales.merge(
    products[['SKU', 'Sub-Category', 'Brand Line', 'Season Tag']], 
    on='SKU', how='left'
)

# Then aggregate T-shirts by brand and month
tshirt_brand = (
    sales_full[sales_full['Sub-Category'] == 'T-Shirts']
    .groupby(['Month', 'Brand Line'])['Units Sold']
    .sum()
    .reset_index()
)

tshirt_brand.columns = ['Month', 'Brand', 'Units_Sold']
tshirt_brand = tshirt_brand.sort_values(['Brand', 'Month']).reset_index(drop=True)

print(tshirt_brand)

KeyError: 'Sub-Category'

In [ ]:
# Create sales_full first
sales_full = sales.merge(
    products[['SKU', 'Sub-Category', 'Brand Line', 'Season Tag']], 
    on='SKU', how='left'
)

# Check columns after merge
print(sales_full.columns.tolist())

In [ ]:
# Aggregate T-shirts by brand and month using correct column name
tshirt_brand = (
    sales_full[sales_full['Sub-Category_y'] == 'T-Shirts']
    .groupby(['Month', 'Brand Line'])['Units Sold']
    .sum()
    .reset_index()
)

tshirt_brand.columns = ['Month', 'Brand', 'Units_Sold']
tshirt_brand = tshirt_brand.sort_values(['Brand', 'Month']).reset_index(drop=True)

print(tshirt_brand)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

def build_features(df):
    df = df.copy()
    df['lag_1'] = df['Units_Sold'].shift(1).fillna(0)
    df['lag_2'] = df['Units_Sold'].shift(2).fillna(0)
    df['is_sale_month'] = df['Month'].isin([1, 2, 6]).astype(int)
    return df

results = {}

for brand in ['CasualCo', 'FlexDenim']:
    brand_df = tshirt_brand[tshirt_brand['Brand'] == brand].copy()
    brand_df = build_features(brand_df)
    
    X = brand_df[['Month', 'lag_1', 'lag_2', 'is_sale_month']].values
    y = brand_df['Units_Sold'].values
    
    X_train, X_test = X[:5], X[5:]
    y_train, y_test = y[:5], y[5:]
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    results[brand] = {
        'actual': y_test[0],
        'predicted': round(y_pred[0]),
        'mae': round(mae)
    }
    
    print(f"=== {brand} ===")
    print(f"Actual June:    {y_test[0]} units")
    print(f"Predicted June: {round(y_pred[0])} units")
    print(f"MAE:            {round(mae)} units")
    print()

## Model Finding — Brand Level Regression
- FlexDenim MAE: 277 units (27% error) — acceptable
- CasualCo MAE: 653 units (188% error) — unacceptable
- CasualCo has extreme EOSS spike that confuses the model
- Root cause: only 6 data points, spike dominates learning
- Decision: CasualCo needs Prophet/ARIMA to handle 
  seasonality properly — simple regression insufficient

- weekend effect is real since sat and sun have 10% more sales, day_of_week / is_weekend feature will be heplful
- june is the worst month, model must prioritise pre sale reordering
- any discount lifts demand, is_discount feature needed
- promotional periods cause demand spikes, is_sale_month feature is required
- prioritise products that actually drive revenue
- 48 SKUs sell on fewer than 30 days — too sparse for 
  individual modelling, needs category level approach

In [ ]:
# Calculate average discount per brand per month
discount_by_brand = (
    sales_full[sales_full['Sub-Category_y'] == 'T-Shirts']
    .groupby(['Month', 'Brand Line'])['Discount %']
    .mean()
    .reset_index()
)

discount_by_brand.columns = ['Month', 'Brand', 'Avg_Discount']
print(discount_by_brand)

In [ ]:
# Merge discount into tshirt_brand
tshirt_brand_v2 = tshirt_brand.merge(
    discount_by_brand, on=['Month', 'Brand'], how='left'
)

print(tshirt_brand_v2)

In [ ]:
# Merge discount into tshirt_brand
tshirt_brand_v2 = tshirt_brand.merge(
    discount_by_brand, on=['Month', 'Brand'], how='left'
)

print(tshirt_brand_v2)

In [ ]:
def build_features_v2(df):
    df = df.copy()
    df['lag_1'] = df['Units_Sold'].shift(1).fillna(0)
    df['lag_2'] = df['Units_Sold'].shift(2).fillna(0)
    df['is_sale_month'] = df['Month'].isin([1, 2, 6]).astype(int)
    return df

print("=== Model v2 — with discount feature ===")
print()

for brand in ['CasualCo', 'FlexDenim']:
    brand_df = tshirt_brand_v2[tshirt_brand_v2['Brand'] == brand].copy()
    brand_df = build_features_v2(brand_df)
    
    # Now includes Avg_Discount as extra feature
    X = brand_df[['Month', 'lag_1', 'lag_2', 
                  'is_sale_month', 'Avg_Discount']].values
    y = brand_df['Units_Sold'].values
    
    X_train, X_test = X[:5], X[5:]
    y_train, y_test = y[:5], y[5:]
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    
    print(f"=== {brand} ===")
    print(f"Actual June:    {y_test[0]} units")
    print(f"Predicted June: {round(y_pred[0])} units")
    print(f"MAE v2:         {round(mae)} units")
    print(f"MAE v1:         {results[brand]['mae']} units")
    
    if mae < results[brand]['mae']:
        print(f"Improved by {results[brand]['mae'] - round(mae)} units")
    else:
        print(f"Got worse by {round(mae) - results[brand]['mae']} units")
    print()

## Model v2 Results — Discount Feature Added
- FlexDenim: MAE improved 277 → 24 units (91% improvement)
  Discount feature added genuine signal for FlexDenim
- CasualCo: MAE worsened 653 → 1,642, predicted negative units
  Classic overfitting — 5 features, 5 training points
  Too many parameters relative to data available
- Decision: use v2 for FlexDenim, revert to v1 for CasualCo
- CasualCo confirmed needs Prophet — regression fundamentally 
  insufficient with this data volume

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet

xl = pd.ExcelFile('/Users/headofthetable/inventory-optimisation/data/raw/Apparel_Store_Inventory_JIT_Dataset.xlsx')
sales = pd.read_excel(xl, 'Daily_Sales_Transactions', header=3)
products = pd.read_excel(xl, 'Product_Master', header=3)

sales['Date'] = pd.to_datetime(sales['Date'])
sales = sales.sort_values('Date').reset_index(drop=True)

print("Loaded:", sales.shape)

In [ ]:
# Simple baseline — average of stable period
stable_period = casualco[casualco['ds'] >= '2026-04-01']
simple_avg = stable_period['y'].mean()
simple_mae = mean_absolute_error(test['y'], 
             [simple_avg] * len(test))

print(f"Simple average forecast: {simple_avg:.1f} units/day")
print(f"Simple average MAE: {simple_mae:.1f} units/day")
print(f"That's {simple_mae * 30:.0f} units error per month")